# Fine-tune IndicF5 on my Marathi narration voice

**What this notebook does, top to bottom:**
1. Installs F5-TTS + Whisper.
2. Takes my 8 clean 24 kHz mono narration files (attached as a Kaggle Dataset).
3. Uses Whisper (large-v3) to auto-segment each recording into 3-12 s clips WITH transcripts -> builds `metadata.csv`.
4. Downloads the IndicF5 base model, converts it to an F5-TTS checkpoint, reading its architecture from `config.json` automatically.
5. Fine-tunes on my voice, saving checkpoints to `/kaggle/working`.
6. Generates a test narration so I can hear the result.

---
### Before running, do these 4 things
1. **Turn on the GPU:** right sidebar -> Session options -> Accelerator -> **GPU T4 x1**.
2. **Accept IndicF5 terms:** visit https://huggingface.co/ai4bharat/IndicF5 (logged in) and click Agree. Create a token at https://huggingface.co/settings/tokens.
3. **Add the token as a secret:** Add-ons -> Secrets -> new secret named **`HF_TOKEN`**.
4. **Attach the dataset:** Add Input -> your uploaded dataset (with `wav_clean/` + `transcripts/`). Note its `/kaggle/input/...` path and set `DATASET_DIR` in the config cell.


## 1 - Install dependencies (~3-4 min)

In [ ]:
%%capture
!pip install -q faster-whisper soundfile soxr huggingface_hub safetensors
!git clone -q https://github.com/SWivid/F5-TTS.git /kaggle/working/F5-TTS
%cd /kaggle/working/F5-TTS
!pip install -q -e .
print("deps installed")

## 2 - Config: paths and knobs

In [ ]:
import os
# >>> EDIT to match your attached dataset path (see the /kaggle/input listing) <<<
DATASET_DIR = "/kaggle/input/marathi-voice/marathi_tts_project"   # contains wav_clean/ and transcripts/
WAV_DIR     = os.path.join(DATASET_DIR, "wav_clean")
TX_DIR      = os.path.join(DATASET_DIR, "transcripts")

WORK        = "/kaggle/working"
DATA_OUT    = f"{WORK}/data/marathi_voice"     # segmented clips + metadata.csv
CLIP_MIN, CLIP_MAX = 3.0, 12.0
WHISPER_MODEL = "large-v3"
DATASET_NAME  = "marathi_voice"

# training knobs (T4 16GB friendly)
BATCH_FRAMES = 3200     # lower to 2000 if OOM
EPOCHS       = 120      # small dataset -> keep BEST checkpoint, not last
SAVE_PER_UPD = 300
LR           = 1e-5

os.makedirs(DATA_OUT, exist_ok=True)
print("wav files:", sorted(os.listdir(WAV_DIR)))
print("transcripts:", sorted(os.listdir(TX_DIR)))

## 3 - Segment + transcribe with Whisper -> metadata.csv

Runs Whisper with word timestamps, then packs words into 3-12 s clips that break on natural pauses. Output: `wavs/*.wav` + `metadata.csv` (`audio_file|text`).

The `transcripts/` you shipped are full-story ground truth; Whisper here makes the clip-level text the trainer needs. Whisper's Marathi is good but not perfect - after the run, skim `metadata.csv` and fix obvious clip errors for best quality.

In [ ]:
import soundfile as sf, soxr, numpy as np, glob
from faster_whisper import WhisperModel

os.makedirs(f"{DATA_OUT}/wavs", exist_ok=True)
model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")

def load24k(p):
    x, sr = sf.read(p, dtype="float32", always_2d=True)
    x = x.mean(1) if x.shape[1] > 1 else x[:,0]
    if sr != 24000: x = soxr.resample(x, sr, 24000, quality="VHQ")
    return x

rows=[]; clip_id=0; total=0.0
for wav in sorted(glob.glob(f"{WAV_DIR}/*.wav")):
    base=os.path.splitext(os.path.basename(wav))[0]
    audio=load24k(wav)
    segments,_=model.transcribe(wav, language="mr", word_timestamps=True,
                                vad_filter=True, beam_size=5)
    words=[w for s in segments for w in (s.words or [])]
    i=0
    while i < len(words):
        j=i; start=words[i].start
        while j < len(words):
            dur=words[j].end - start
            gap=(words[j+1].start - words[j].end) if j+1 < len(words) else 1.0
            if dur >= CLIP_MIN and (gap > 0.35 or dur >= CLIP_MAX): break
            j+=1
        end=words[min(j,len(words)-1)].end
        text=" ".join(w.word.strip() for w in words[i:j+1]).strip()
        i=j+1
        if end-start < 1.0 or len(text) < 5: continue
        a=int(max(0,start-0.05)*24000); b=int(min(len(audio)/24000,end+0.15)*24000)
        clip=audio[a:b]
        name=f"{base}_{clip_id:04d}.wav"; clip_id+=1
        sf.write(f"{DATA_OUT}/wavs/{name}", clip, 24000, subtype="PCM_16")
        rows.append((f"wavs/{name}", text)); total+=(b-a)/24000

with open(f"{DATA_OUT}/metadata.csv","w",encoding="utf-8",newline="") as f:
    f.write("audio_file|text\n")
    for a,t in rows: f.write(f"{a}|{t}\n")
print(f"{len(rows)} clips, {total/60:.1f} min total")
for r in rows[:5]: print("  ", r[1][:70])

## 4 - Build the F5-TTS arrow dataset

In [ ]:
%cd /kaggle/working/F5-TTS
!python src/f5_tts/train/datasets/prepare_csv_wavs.py {DATA_OUT} data/{DATASET_NAME}_char
!ls -la data/{DATASET_NAME}_char

## 5 - Download IndicF5 and convert to an F5-TTS checkpoint

Reads IndicF5's `config.json` at runtime and prints the architecture (so nothing is guessed). Converts `model.safetensors` into the `ema_model_state_dict` `.pt` format the finetune CLI expects, and uses IndicF5's own `vocab.txt`.

In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import json, torch
from safetensors.torch import load_file

tok = UserSecretsClient().get_secret("HF_TOKEN")
repo = "ai4bharat/IndicF5"
cfg_path   = hf_hub_download(repo, "config.json",           token=tok)
vocab_path = hf_hub_download(repo, "checkpoints/vocab.txt", token=tok)
weights    = hf_hub_download(repo, "model.safetensors",     token=tok)

cfg = json.load(open(cfg_path))
print("IndicF5 config.json (first 1500 chars):")
print(json.dumps(cfg, indent=2)[:1500])
print("\nvocab size:", sum(1 for _ in open(vocab_path, encoding="utf-8")))

sd = load_file(weights)
print("\nfirst 10 weight keys:", list(sd.keys())[:10])
# F5-TTS finetune loads ckpt['ema_model_state_dict']; ensure the ema_model. prefix.
ema = {(k if k.startswith("ema_model.") else f"ema_model.{k}"): v for k,v in sd.items()}
os.makedirs(f"{WORK}/base", exist_ok=True)
BASE_CKPT = f"{WORK}/base/indicf5_base.pt"
torch.save({"ema_model_state_dict": ema, "update": 0, "step": 0}, BASE_CKPT)
BASE_VOCAB = f"{WORK}/base/vocab.txt"
open(BASE_VOCAB,"w",encoding="utf-8").write(open(vocab_path,encoding="utf-8").read())
print("saved base ckpt ->", BASE_CKPT)
print("\nNOTE: if the 'first 10 weight keys' above are NOT bare model keys, adjust the ema remap accordingly.")

## 6 - Fine-tune

IndicF5 uses the F5TTS_Base architecture family. If the printed config shows a v1 arch, switch `--exp_name` to `F5TTS_v1_Base`. With ~1 hr of data, keep the BEST-sounding checkpoint (small datasets overfit past a point).

In [ ]:
%cd /kaggle/working/F5-TTS
# match token ids to the base model by using IndicF5's vocab
!cp {WORK}/base/vocab.txt data/{DATASET_NAME}_char/vocab.txt

!f5-tts_finetune-cli \
  --exp_name F5TTS_Base \
  --dataset_name {DATASET_NAME} \
  --pretrain {WORK}/base/indicf5_base.pt \
  --tokenizer char \
  --learning_rate {LR} \
  --batch_size_per_gpu {BATCH_FRAMES} \
  --batch_size_type frame \
  --epochs {EPOCHS} \
  --save_per_updates {SAVE_PER_UPD} \
  --last_per_updates {SAVE_PER_UPD} \
  --finetune \
  --logger tensorboard

## 7 - Test the fine-tuned voice

In [ ]:
import glob, csv
ckpts=sorted(glob.glob(f"/kaggle/working/F5-TTS/ckpts/{DATASET_NAME}/*.pt"))
print("checkpoints:", [os.path.basename(c) for c in ckpts])
CKPT=ckpts[-1]  # also try earlier ones; keep the best

meta=list(csv.reader(open(f"{DATA_OUT}/metadata.csv",encoding="utf-8"),delimiter="|"))[1:]
ref_rel,ref_txt=meta[0]                 # pick a clean ~10 s reference clip
REF=os.path.join(DATA_OUT, ref_rel)
GEN_TEXT="ही एक नवीन मराठी भयकथा आहे. रात्रीच्या अंधारात, त्या जुन्या वाड्यात, काहीतरी हालचाल जाणवत होती."

!f5-tts_infer-cli \
  --model F5TTS_Base \
  --ckpt_file "{CKPT}" \
  --vocab_file {WORK}/base/vocab.txt \
  --ref_audio "{REF}" \
  --ref_text "{ref_txt}" \
  --gen_text "{GEN_TEXT}" \
  --output_dir {WORK}/test_out
from IPython.display import Audio, display
for f in glob.glob(f"{WORK}/test_out/*.wav"): print(f); display(Audio(f))

## 8 - Save your model
Checkpoints are in `/kaggle/working/F5-TTS/ckpts/marathi_voice/`.
- Click **Save Version** to persist, or download the best `.pt` + `vocab.txt` from the Output tab.
- Bring both to your PC for local inference (fits a GTX 1650 in fp16).